In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [3]:
load_dotenv()


True

In [9]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model='gpt-3.5-turbo',max_tokens=500)

In [10]:
# Carregar o PDF
pdf_link = "./data/visao-estereo-rev.pdf"

loader = PyPDFLoader(pdf_link,extract_images=False)
pages = loader.load_and_split()

len(pages)

16

In [11]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

In [12]:
# Storages o child vai buscar o db e o parent fica em memoria

store = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings, persist_directory="childVectorDB")

In [13]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_document_retriever.add_documents(pages,id=None)

In [14]:
parent_document_retriever.vectorstore.get()

{'ids': ['4124bdce-1dd1-4988-bd45-64b208f2d60e',
  'f535c4cf-8f6b-4b2a-9986-f9166e26d953',
  '491e9db8-e2dd-4369-b768-3173fab9bac0',
  '00cc4acd-76ad-4dc8-9f98-d61a206d6018',
  '811debb4-3aef-4b3b-9000-072c8f298c1b',
  '6ba26389-75b1-4b6d-9699-6aa847c2ba30',
  '05ab4327-d111-4c9f-87bd-4cdda54ff3a3',
  '75f21166-9243-403d-ba46-06e18ed3b157',
  'b72f4c98-e1cb-4963-88ef-e604baaea200',
  '755fafaa-e8c9-4583-95a9-465c99620fee',
  '35c0d460-fc48-42f6-b5eb-ead11abe8dc0',
  'fa8cbbc3-a516-40f2-9642-4686f0418673',
  'fa8fa0f7-a984-4068-83c3-a1fb1cc4fa2a',
  '85fdfd96-ba17-4787-a4df-7883fc4c3156',
  'e008ef0e-209c-4e2c-b523-b7e2067168d8',
  'dfc23c7b-73d4-45ab-a82e-5baf49192e5c',
  'a29e5269-c090-43ba-ac51-bc3d4f409789',
  'ef5f7595-a79c-43bc-ac96-cbff9ff6a984',
  '04fd17da-d11a-4876-9b22-409543ab6e69',
  '8b47655f-ae67-4fe8-91bb-b1c8be7473c4',
  '127e5586-3e6f-498a-a401-0899807d906d',
  '8b8cbfed-fe1d-4554-96e0-139e39f80f51',
  '1bd38cdc-4d67-475e-a487-02b9e092eb1d',
  '73edb2d0-6f09-4807-bf57-

In [20]:
TEMPLATE = """
    Você é um especialista em visão computacional para a parte de visão estéreo. Responda a pergunta utilizando o contexto informado
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [22]:
setup_retrival = RunnableParallel({"question": RunnablePassthrough(), "context": parent_document_retriever})
output_parser = StrOutputParser()


In [23]:
parent_chain_retrival = setup_retrival | rag_prompt | llm | output_parser

In [24]:
parent_chain_retrival.invoke("Quais são os principais conceitos de visão estéreo?")

'Os principais conceitos de visão estéreo incluem a disparidade, que é a diferença entre as posições dos objetos nas imagens das duas câmeras, a distância física entre o observador e o objeto (distância), e a distância subjetiva ao objeto percebida pelo observador (profundidade). Além disso, o estudo de visão estéreo é dividido em duas partes: medindo a disparidade e usando-a para obter informações de profundidade. O problema da correspondência é um dos desafios da visão estéreo, que envolve encontrar os pontos correspondentes nas imagens das duas câmeras.'